## BronzeWork Incremental
Incremental Bronze ingestion with rerun-safe watermark logic.

### Step 1: Imports and Setup

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:

bronze_uuid = str(uuid.uuid4())
print(f" Current bronze run ID: {bronze_uuid}")

### Step 2: Helper functions
This cell contains reusable functions:
- `get_last_successful_watermark()`: Reads the last processed watermark from the control table.
- `upsert_bronze_control()` Updates the control table after a successful Bronze load.

These functions keep the main load logic cleaner and easier to understand.

In [0]:
def get_last_successful_watermark(tableId: int):
    ingestion_control_df = (
        spark.read.table("bronze.control.ingestion_control")
        .filter(
            (F.col("table_id") == F.lit(tableId))
            & (F.col("run_status") == F.lit("success"))
        )
        .orderBy(F.col("updated_at").desc())
        .limit(1)
    )

    if ingestion_control_df.isEmpty():
        return None, None

    latest_row = ingestion_control_df.collect()

    return latest_row[0]["last_successful_ts"], latest_row[0]["last_successful_pk"]

In [0]:
def upsert_bronze_control(
    run_id: str,
    tableId: int,
    last_successful_ts: datetime,
    last_successful_pk: int,
    rows_written: int,
    status: str,
):
    data = [
        (
            run_id,
            tableId,
            last_successful_ts,
            last_successful_pk,
            int(rows_written),
            status,
            datetime.utcnow(),
        ),
    ]
    schema = """
    run_id STRING,
    table_id BIGINT,
    last_successful_ts TIMESTAMP,
    last_successful_pk BIGINT,
    rows_written BIGINT,
    run_status STRING,
    updated_at TIMESTAMP
    """

    control_df = spark.createDataFrame(data, schema)
    delta_table = DeltaTable.forName(spark, "bronze.control.ingestion_control")

    delta_table.alias("t").merge(
        control_df.alias("s"), "t.table_id = s.table_id and t.run_id = s.run_id"
    ).whenMatchedUpdate(
        set={
            "t.last_successful_ts": "s.last_successful_ts",
            "t.last_successful_pk": "s.last_successful_pk",
            "t.rows_written": "s.rows_written",
            "t.run_status": "s.run_status",
            "t.updated_at": "s.updated_at",
        }
    ).whenNotMatchedInsertAll().execute()

### Step 3: Bronze incremental load loop
This is the main Bronze logic.

For each table, the notebook:
- Reads the last watermark
- Reads the source SQL table
- Filters only new / changed rows
- Adds Bronze audit columns
- Appends the rows into the Bronze Delta table
- Updates the control table

This is the core incremental loading logic.

In [0]:
object_list = (
    spark.read.table("bronze.control.object_list")
    .filter(F.col("layer") == F.lit("bronze"))
    .collect()
)

for row in object_list:
    table_id = row["table_id"]
    table_name = row["table_name"]
    ts_col = row["ts_col"]
    pk_col = row["pk_col"]
    
    source_table = f"`source`.landing.{table_name}"
    target_table = f"`bronze`.raw.{table_name}_raw"

    print(f"Processing table: {table_name}")

    source_df = spark.read.table(source_table).withColumn(
        ts_col, F.col(ts_col).cast("timestamp")
    )

    last_successful_ts, last_successful_pk = get_last_successful_watermark(table_id)

    if last_successful_ts is None:
        rows_to_load = source_df
        print(f"Initial load for table {table_name}")
    else:
        rows_to_load = source_df.filter(
            (F.col(ts_col) > F.lit(last_successful_ts))
            | (
                (F.col(ts_col) == F.lit(last_successful_ts))
                & (F.col(pk_col).cast("long") > F.lit(int(last_successful_pk)))
            )
        )
        print(
            f"Incremental load for table {table_name} since ts={last_successful_ts}, pk={last_successful_pk}"
        )

    cols = {
        "bronze_source_table": F.lit(table_name),
        "bronze_ingested_at": F.current_timestamp(),
        "bronze_run_id": F.lit(bronze_uuid),
    }

    rows_load = rows_to_load.withColumns(cols)

    rows_count = rows_load.count()
    print(f"Rows to load for table {table_name}: {rows_count}")


    if rows_count == 0:
        print(f"No new records to load for table {table_name}")
        upsert_bronze_control(
            run_id=bronze_uuid,
            tableId=table_id,
            last_successful_ts=last_successful_ts,
            last_successful_pk=last_successful_pk,
            rows_written=rows_count,
            status="success"
        )
        continue

    rows_load.write.format("delta").mode("append").saveAsTable(target_table)
    print(f"Appended {rows_count} rows to {target_table}")

    max_ts = rows_load.agg(F.max(ts_col).alias("max_ts")).collect()[0]["max_ts"]

    max_pk = (
        rows_load.filter(F.col(ts_col) == F.lit(max_ts))
        .agg(F.max(pk_col).cast("long").alias("max_pk"))
        .collect()[0]["max_pk"]
    )

    upsert_bronze_control(
        run_id=bronze_uuid,
        tableId=table_id,
        last_successful_ts=max_ts,
        last_successful_pk=max_pk,
        rows_written=rows_count,
        status="success"

    )
    print(
        f"Updated control table for {table_name}: max_ts={max_ts}, max_pk={max_pk}, rows_written={rows_count}"
    )
     

In [0]:
%sql
SELECT * FROM bronze.control.ingestion_control